# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [10]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [11]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    # TODO: Implement a tokenization strategy
    B,C,H,W=x.shape
    assert H % patch_size == 0 and W % patch_size == 0, "H/P and W/P must be integers"
    h=H//patch_size
    w=W//patch_size
    x = x.view(B, C, h, patch_size, w, patch_size)
    x.permute(0,2,4,1,3,5).contiguous()

    return x.reshape(B,h*w,C*patch_size*patch_size)

In [12]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.patch_dim=patch_dim
        self.d_model=d_model
        self.proj = nn.Linear(patch_dim,d_model)


    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.proj(x_patches) 


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        # TODO: implement
        self.pos_embedding=nn.Parameter(torch.zeros(1,num_tokens,d_model))
        nn.init.trunc_normal_(self.pos_embedding,std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.pos_embedding+x

In [13]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: implement
        layers=[]
        layers.append(nn.Linear(d_model,d_ff))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(p=dropout)) #0.1 in the paper
        layers.append(nn.Linear(d_ff,d_model))
        layers.append(nn.Dropout(p=dropout))
        self.net=nn.Sequential(*layers)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.net(x)


class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        # TODO: implement
        self.variant=variant
        self.wterm=nn.Linear(d_model,d_ff_gated)
        self.vterm=nn.Linear(d_model,d_ff_gated)
        self.out=nn.Linear(d_ff_gated,d_model)
        self.dropout=nn.Dropout(dropout)
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        w=self.wterm(x)
        v=self.vterm(x)
        if self.variant == 'geglu':
            term=F.gelu(w)
        elif self.variant == 'swiglu':
            term=F.silu(w)
        elif self.variant =='reglu':
            term=F.relu(w)
        elif self.variant == 'bilinear':
            term=w
        else:
            term=F.sigmoid(w)

        x=term*v
        x=self.dropout(x)
        return self.dropout(self.out(x))

In [14]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        # TODO: implement. For attention use nn.MultiHeadAttention 
        self.ln1=nn.LayerNorm(d_model)
        self.ln2=nn.LayerNorm(d_model)

        self.attention=nn.MultiheadAttention(d_model,n_heads,dropout,batch_first=True)
        self.mlp=mlp
        self.dropout=nn.Dropout(dropout)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        norm_x = self.ln1(x)
        attention_out, _ = self.attention(norm_x, norm_x, norm_x)
        x = x + self.dropout(attention_out)
        x = x + self.dropout(self.mlp(self.ln2(x)))
        
        return x 

In [15]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size
        d_ff_gated=int(d_ff*2/3)
        

        # TODO: implement a strategy for embedding the patches
        self.patch_embedded=PatchEmbed(patch_dim,d_model)
        self.positional_embedded=PositionalEmbedding(self.num_tokens,d_model)
                

        # TODO: implement a strategy to select the right mlp version for your experiment
        def choose_mlp(mlp_kind):
            if mlp_kind =='FFN':
                return FeedForward(d_model,d_ff,dropout)
            else:
                return GLUFeedForward(d_model,d_ff_gated,dropout,mlp_kind)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=choose_mlp(mlp_kind), # TODO: Feed your mlp to the encoder blocks
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        # TODO: Add a head to project to the amount of output classes you have
        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement
        x=patchify(x,self.patch_size)
        x=self.patch_embedded(x)
        x=self.positional_embedded(x)
        for block in self.blocks:
            x = block(x)
            
        x = x.mean(dim=1)        
        x = self.ln_final(x)
        logits = self.head(x)
        
        return logits

In [16]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [17]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []
    steps: list[int] = []
    global_step = 0

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        epoch_loss=0.0
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            criterion=nn.CrossEntropyLoss()
            loss =criterion(logits,yb) # TODO: Your criterion

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())
            global_step += 1
            epoch_loss += loss.item()

            if global_step % 100 == 0:
                print(f"  Step {global_step} | Loss: {loss.item():.4f}")

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")

    return {
        # TODO: Return your metrics that you think will support your claim for this experiment
        # "kind": mlp_kind,
        # "losses": train_losses,
        # "accuracies": test_accs,
        # "total_steps": len(train_losses),
        # "params": sum(p.numel() for p in model.parameters() if p.requires_grad)
    }

In [18]:
# import matplotlib.pyplot as plt
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="cpu")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

runs = ['geglu','reglu','FFN','swiglu','bilinear'] # TODO: Name your runs
results = []

for kind in runs:
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    # TODO: print anything you might want here
    print(f"\nRun: {kind} | " )
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)
#fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# for res in results:
#     kind = res["kind"]
    
#     steps = range(len(res["losses"]))
#     ax1.plot(steps, res["losses"], label=f"{kind} (Loss)", alpha=0.7)
    
#     epochs = range(1, len(res["accuracies"]) + 1)
#     ax2.plot(epochs, res["accuracies"], label=f"{kind}", marker='o')

# ax1.set_xlabel("Optimization steps")
# ax1.set_ylabel("Cross Entropy Loss")
# ax1.legend()
# ax1.grid(True, linestyle='--', alpha=0.6)

# ax2.set_xlabel("Epoch")
# ax2.set_ylabel("Accuracy")
# ax2.legend()
# ax2.grid(True, linestyle='--', alpha=0.6)

# plt.tight_layout()
# plt.show()



Run: geglu | 
  Step 100 | Loss: 0.4061
  Step 200 | Loss: 0.3355
  Step 300 | Loss: 0.2437
  Step 400 | Loss: 0.1682
[geglu] epoch 1/5 | test acc: 0.9422
  Step 500 | Loss: 0.2251
  Step 600 | Loss: 0.1933
  Step 700 | Loss: 0.2863
  Step 800 | Loss: 0.3228
  Step 900 | Loss: 0.0949
[geglu] epoch 2/5 | test acc: 0.9565
  Step 1000 | Loss: 0.1136
  Step 1100 | Loss: 0.1454
  Step 1200 | Loss: 0.1900
  Step 1300 | Loss: 0.0721
  Step 1400 | Loss: 0.1056
[geglu] epoch 3/5 | test acc: 0.9676
  Step 1500 | Loss: 0.2334
  Step 1600 | Loss: 0.0900
  Step 1700 | Loss: 0.0841
  Step 1800 | Loss: 0.1517
[geglu] epoch 4/5 | test acc: 0.9691
  Step 1900 | Loss: 0.1531
  Step 2000 | Loss: 0.0792
  Step 2100 | Loss: 0.0543
  Step 2200 | Loss: 0.1488
  Step 2300 | Loss: 0.0527
[geglu] epoch 5/5 | test acc: 0.9739

Run: reglu | 
  Step 100 | Loss: 0.3828
  Step 200 | Loss: 0.3776
  Step 300 | Loss: 0.3067
  Step 400 | Loss: 0.3436
[reglu] epoch 1/5 | test acc: 0.9476
  Step 500 | Loss: 0.1671
  Step